# 香港特区水环境监测站历史数据转换: <br>`csv`读取和`sqlite`数据库导出

## 数据来源: 
数据存档于"香港特别行政区开放数据平台" ([https://data.gov.hk](https://data.gov.hk)), 本项目使用以下数据集的数据: 
```
[1] ENVIRONMENTAL PROTECTION DEPARTMENT. Historical Marine Water Quality 
    Data (English Version)[EB/OL]. (2025-08-24). 
    https://data.gov.hk/en-data/dataset/hk-epd-marineteam-marine-water-
    quality-historical-data-en
    
[2] ENVIRONMENTAL PROTECTION DEPARTMENT. Historical River Water Quality 
    Data (English Version)[EB/OL]. (2025-08-24). 
    https://data.gov.hk/en-data/dataset/hk-epd-riverteam-river-water-
    quality-historical-data-en
```
上述两个数据集在该平台的发布和更新由特区环境保护部门负责. 该部门不定期在开放数据平台发布数据的更新版本, 本项目使用平台存档的2025年8月24日版本. 

In [1]:
import io, sys, os; 
nb_dir = os.getcwd(); 
sys.path.append(nb_dir); 

In [2]:
import csv, sqlite3; 
import datetime; 

In [3]:
import re; 
import itertools as it, collections as coll; 

## 指定`csv`文件的名称和所在路径
数据为逗号分隔值 (`csv`) 格式文件, 采用`UTF-8`格式字符编码. 

`csv`文件以年为单位划分. 每个`csv`文件包括香港特区所有河流水质 (或海洋水质) 监测站在**某个特定公历年**内 (1月1日至12月31日) 的所有水样化验记录. 

即使在同一公历年内, 河流监测站和海洋监测站的水质数据分别由不同的`csv`文件记录 (*id est* 任何一个`csv`文件包含河流监测站的水质数据时, 必然不含海洋监测站的水质数据. *vice versa*). \
主要原因: 河流水样和海洋水样分析的指标内容不同$^{[1, 2]}$. 

参考文献: 

```
[1] ENVIRONMENTAL PROTECTION DEPARTMENT. Data Specification for Historical 
    Marine Water Quality Data (English Version)[EB/OL]. (2025-08-17). 
    https://historical-resource-archive.oss-cn-hongkong.aliyuncs.com/
    90fcbc021e4c1778e8a0eee81a3baaf6d7e8643302e0f3c44f6f9561b24fe235/
    data-dictionary/2025/08/20250817-historical_marine_data_dictionary
    _en.pdf
    
[2] ENVIRONMENTAL PROTECTION DEPARTMENT. Data Specification for Historical 
    River Water Quality Data (English Version)[EB/OL]. (2025-08-17). 
    https://historical-resource-archive.oss-cn-hongkong.aliyuncs.com/
    ffa1fb5f1c503f35729f6b0f3c5848e0b26b75cd79a93e8023ecd1ccc67728c1/
    data-dictionary/2025/08/20250817-historical_river_data_dictionary
    _en.pdf
```

### 路径定位
文件使用相对路径

In [4]:
wq_spot_value_path = os.sep.join(
    [nb_dir] + [os.pardir] * 3 + ["Source", "In-Situ_Monitoring"]
); 
wq_spot_value_path = os.path.normpath(wq_spot_value_path); 

### 文件名匹配
文件名称格式: \
海洋监测站: `更新日期八位数码-版本数码号-historical-marine-监测年度四位数码-en.csv`; \
河流监测站: `更新日期八位数码-版本数码号-historical-river-监测年度四位数码-en.csv`

据此构建正则表达式, 在相对路径的目录下搜索并匹配符合条件的文件名. 

In [5]:
csv_ver_date_str = "20250824"; 
file_mar_filter, file_riv_filter = tuple(re.compile(
    r"{ver}-\d+-{aq}-historical-(?P<year>\d{{4}})-en\.csv".format(
        ver=csv_ver_date_str, aq=aq
    )
) for aq in ("marine", "river"))

In [6]:
wq_mar_spot_filename_csv = dict(); 
wq_riv_spot_filename_csv = dict(); 
for cur_dir, sub_dirs, sub_files in os.walk(wq_spot_value_path, topdown=True): 
    for file in sub_files: 
        match = file_mar_filter.match(file); 
        if match: 
            wq_mar_spot_filename_csv.update(
                {match.groupdict()["year"]: match.group(0)}
            ); 
        match = file_riv_filter.match(file); 
        if match: 
            wq_riv_spot_filename_csv.update(
                {match.groupdict()["year"]: match.group(0)}
            ); 
    break; 

In [7]:
wq_riv_spot_value_csv = tuple(
    os.sep.join(
        [wq_spot_value_path, wq_riv_spot_filename_csv[str(yr)]]
    ).format(year=yr) for yr in range(2015, 2025)
); 

wq_mar_spot_value_csv = tuple(
    os.sep.join(
        [wq_spot_value_path, wq_mar_spot_filename_csv[str(yr)]]
    ).format(year=yr) for yr in range(2015, 2025)
); 

## `csv`文件的打开与读取
打开每个`csv`文件并读取, 其中第一行为表头, 声明每个字段的名称; 后续行均为记录, 每条记录是特定年份下单个监测站在特定单日的各水质指标监测数据. 

In [8]:
wq_riv_fields = list(); wq_riv_items = list(); 
for file in wq_riv_spot_value_csv: 
    with io.open(file, mode="r", encoding="utf-8") as strm_csv: 
        handle_csv = csv.reader(strm_csv); 
        wq_riv_fields.append(next(handle_csv)); 
        for rec in handle_csv: 
            wq_riv_items.append(rec); 

wq_mar_fields = list(); wq_mar_items = list(); 
for file in wq_mar_spot_value_csv: 
    with io.open(file, mode="r", encoding="utf-8") as strm_csv: 
        handle_csv = csv.reader(strm_csv); 
        wq_mar_fields.append(next(handle_csv)); 
        for rec in handle_csv: 
            wq_mar_items.append(rec); 

### 字段特征分析

分析每个csv文件的字段名称和顺序是否相同

In [9]:
print(all(fld == wq_riv_fields[0] for fld in wq_riv_fields[1: ])); 
print(all(fld == wq_mar_fields[0] for fld in wq_mar_fields[1: ])); 

True
True


所有河流水质`csv`文件, 以及所有海洋水质`csv`文件的字段, 分别相同

In [10]:
print(wq_riv_fields[0]); 

['\ufeff"Water Control Zone"', 'River', 'Station', 'Dates', 'Sample No', '5-Day Biochemical Oxygen Demand (mg/L)', 'Aluminium (μg/L)', 'Ammonia-Nitrogen (mg/L)', 'Anionic Surfactants (as Manoxol OT) (mg/L)', 'Antimony (μg/L)', 'Arsenic (μg/L)', 'Barium (μg/L)', 'Beryllium (μg/L)', 'Boron (μg/L)', 'Cadmium (μg/L)', 'Chemical Oxygen Demand (mg/L)', 'Chloride (mg/L)', 'Chlorophyll-𝘢 (μg/L)', 'Chromium (μg/L)', 'Conductivity (μS/cm)', 'Copper (μg/L)', 'Cyanide (mg/L)', 'Dissolved Oxygen (%saturation)', 'Dissolved Oxygen (mg/L)', 'Faecal Coliforms (counts/100mL)', 'Flow (m³/s)', 'Fluoride (mg/L)', 'Free Hydrogen Sulphide (mg/L)', 'Iron (μg/L)', 'Lead (μg/L)', 'Manganese (μg/L)', 'Mercury (μg/L)', 'Molybdenum (μg/L)', 'Nickel (μg/L)', 'Nitrate-Nitrogen (mg/L)', 'Nitrite-Nitrogen (mg/L)', 'Oil and Grease (mg/L)', 'Orthophosphate Phosphorus (mg/L)', 'pH', 'Phaeopigment (μg/L)', 'Salinity (psu)', 'Silica (as SiO₂) (mg/L)', 'Silver (μg/L)', 'Sulphide (mg/L)', 'Suspended solids (mg/L)', 'Thallium

In [11]:
print(wq_mar_fields[0]); 

['\ufeff"Water Control Zone"', 'Station', 'Dates', 'Sample No', 'Depth', '5-day Biochemical Oxygen Demand (mg/L)', 'Ammonia Nitrogen (mg/L)', 'Chlorophyll-a (μg/L)', 'Dissolved Oxygen (%saturation)', 'Dissolved Oxygen (mg/L)', 'E. coli (cfu/100mL)', 'Faecal Coliforms (cfu/100mL)', 'Nitrate Nitrogen (mg/L)', 'Nitrite Nitrogen (mg/L)', 'Orthophosphate Phosphorus (mg/L)', 'pH', 'Phaeo-pigments (μg/L)', 'Salinity (psu)', 'Secchi Disc Depth (M)', 'Silica (mg/L)', 'Suspended Solids (mg/L)', 'Temperature (°C)', 'Total Inorganic Nitrogen (mg/L)', 'Total Kjeldahl Nitrogen (mg/L)', 'Total Nitrogen (mg/L)', 'Total Phosphorus (mg/L)', 'Turbidity (NTU)', 'Unionised Ammonia (mg/L)', 'Volatile Suspended Solids (mg/L)']


#### 字段重命名

为了在`sqlite`中通过`SQL`命令便捷地[插入](#csv提取结果合并入库)和访问特定字段下的数据, 需要对字段重命名. 

重命名原则: 
* 字段名称中只包括: 小写拉丁字母`a`-`z`, 数字`0`-`9`, 半角下划线`_` (`U+005f`); 
* 字段名称的首个字符不能为数字; 
* `Unicode`中的"斜体拉丁字母"字符 (用于物种的拉丁学名等场合), 替换为`ASCII`字符集中, 对应的正常拉丁字母; 

重命名流程: 
1. 将字段中包含的"斜体拉丁字母"替换为正常拉丁字母; 
2. 将字段内圆括号及其中的所有标注内容删除; 
3. 将相邻单词间的空格和连字符替换为下划线; 
4. 将字段名称中除限定字符以外的其他字符删除, 并将大写拉丁字母转换为小写; 
5. 经过上述处理后, 如果存在两个以上名称相同的字段, 从第二个同名字段起, 在其结尾添加出现的顺序, 以下划线分隔. 

In [12]:
wq_field_anno_filter = re.compile(r'\(.+?\)'); 
wq_field_italic_rule = {
    src: dest for src, dest in tuple(it.chain(
        zip(range(119860, 119886), range(65, 91)), 
        zip(range(119886, 119912), range(97, 123)), 
        zip((8462, ), (104, )), 
        zip(range(120328, 120354), range(65, 91)), 
        zip(range(120354, 120380), range(97, 123)), 
    ) ) 
}; 
wq_field_word_conn_rule = str.maketrans( {
   src: dest for src, dest in zip("\x20\x2d", "\x5f\x5f")
}) ; 
wq_field_illegal_char_filter = re.compile(r'(\A[0-9])|[\W]'); 

In [13]:
def field_name_norm(name: str) -> str: 
    name_norm = name; 
    name_norm = name_norm.translate(wq_field_italic_rule); 
    name_norm = wq_field_anno_filter.sub("", name_norm); 
    name_norm = name_norm.strip().translate(wq_field_word_conn_rule); 
    name_norm = wq_field_illegal_char_filter.sub("", name_norm); 
    name_norm = name_norm.lower(); 
    return name_norm; 

In [14]:
cntr = coll.Counter(); wq_riv_fields_norm = list(); 
for field in wq_riv_fields[0]: 
    field_norm = field_name_norm(field); 
    cntr.update((field_norm, )); 
    dup = cntr[field_norm]; 
    if dup >= 2: 
        field_norm = "{name}_{dup:d}".format(name=field_norm, dup=dup); 
    wq_riv_fields_norm.append(field_norm); 

In [15]:
cntr = coll.Counter(); wq_mar_fields_norm = list(); 
for field in wq_mar_fields[0]: 
    field_norm = field_name_norm(field); 
    cntr.update((field_norm, )); 
    dup = cntr[field_norm]; 
    if dup >= 2: 
        field_norm = "{name}_{dup:d}".format(name=field_norm, dup=dup); 
    wq_mar_fields_norm.append(field_norm); 

#### 字段追加
在`dates`字段后, 追加三个字段`year_obsv`, `month_obsv`, `day_obsv`
分别存储监测日期的年, 月, 日

In [16]:
wq_riv_fields_addition = wq_riv_fields_norm.copy(); 
idx_fld_dates = wq_riv_fields_addition.index("dates")
for date_unit in ("year", "month", "day"): 
    date_field_name = date_unit + "_obsv"; 
    wq_riv_fields_addition.insert(
        idx_fld_dates + 1, date_field_name
    ); 
    idx_fld_dates += 1; 
wq_riv_fields_norm = wq_riv_fields_addition.copy(); 

In [17]:
wq_mar_fields_addition = wq_mar_fields_norm.copy(); 
idx_fld_dates = wq_mar_fields_addition.index("dates")
for date_unit in ("year", "month", "day"): 
    date_field_name = date_unit + "_obsv"; 
    wq_mar_fields_addition.insert(
        idx_fld_dates + 1, date_field_name
    ); 
    idx_fld_dates += 1; 
wq_mar_fields_norm = wq_mar_fields_addition.copy(); 

#### 标准化处理后字段的差异

分别计算河流水质数据集和海洋水质数据集中独有的字段

In [18]:
set(wq_riv_fields_norm).difference(set(wq_mar_fields_norm))

{'aluminium',
 'anionic_surfactants',
 'antimony',
 'arsenic',
 'barium',
 'beryllium',
 'boron',
 'cadmium',
 'chemical_oxygen_demand',
 'chloride',
 'chromium',
 'conductivity',
 'copper',
 'cyanide',
 'flow',
 'fluoride',
 'free_hydrogen_sulphide',
 'iron',
 'lead',
 'manganese',
 'mercury',
 'molybdenum',
 'nickel',
 'oil_and_grease',
 'phaeopigment',
 'river',
 'silver',
 'sulphide',
 'thallium',
 'total_organic_carbon',
 'total_solids',
 'total_volatile_solids',
 'vanadium',
 'water_temperature',
 'zinc'}

In [19]:
set(wq_mar_fields_norm).difference(set(wq_riv_fields_norm))

{'depth',
 'phaeo_pigments',
 'secchi_disc_depth',
 'temperature',
 'total_inorganic_nitrogen',
 'total_nitrogen',
 'unionised_ammonia',
 'volatile_suspended_solids'}

## 数据入库前的标准化
`sqlite`中, 同一表格内, 不同记录在相同字段的数据类型必须相同. 

为了后续计算的便利, 需要对部分数据进行标准化处理: 
* 日期 (`dates`字段): 原始`csv`中, 日期为`yyyy-mm-dd`的字符形式, 符合`ISO-8601-1`中的日期格式, \
    标准化过程中, 将其转换为自公元1970年01月01日开始的**天数**. 
    > 1970年1月1日为第0天, 1月2日为第1天, 1971年1月1日为第365天, 以此类推. 
    >
    > 此处不使用`Unix`时间戳, 因为该时间戳精确到秒, 而且需要考虑时区问题, \
    > 但水质取样日期只精确到日, 没有具体的时分秒信息. 
    
* 水质测量结果中的空值: 测量结果中的空串, `null`和`n/a`字符串, 在插入数据库表格时, 均以空值论. 

* 水质测量结果中低于检出限的结果处理: 
    * 如果表示结果的文本, 为大于零的整数或小数的字符串形式, 直接按字面量转化为浮点数;  
    * 如果表示结果的文本为"0", 直接转换为浮点数`0.`; 
    * 如果表示结果的文本, 为小于号开头, 后接一个正整数或正小数的字符串 (例如 "<1.0"), 表示测量结果**低于测定方法的检出限**. 
        * 不能直接其视为`0. ` (因为证据不足, 否则在原始数据中就会直接记为"0"); 
        * 它的含义与实测结果中直接出现的零值不同. 
        * 参考2016年至2025年间发表的部分文献所述的方法, 后续计算过程中**按照检出限的$1 \over 2$处理**$^{[1-5]}$

参考文献: 
```
[1] 朱媛媛, 田进军, 李红亮, 等. 丹江口水库水质评价及水污染特征[J]. 
    农业环境科学学报, 2016, 35(01): 139-147.
    
[2] 蔡竹, 李垚垚, 何丽. 石笋沟水质现状评价[J]. 贵阳学院学报(自然
    科学版), 2020, 15(03): 45-47. 
    
[3] 刘光正, 王明森, 刘健, 等. 大明湖水污染物因子分析及富营养化评价[J]. 
    济南大学学报(自然科学版), 2023, 37(06): 696-702. 
    
[4] 陈振明. 汕头市近岸海域水质状况研究[J]. 黑龙江环境通报, 2024, 
    37(08): 13-15. 

[5] 朱纯祥. 龙河口水库水体富营养化评价及其变化趋势研究[J]. 安徽水利
    水电职业技术学院学报, 2025, 25(03): 28-31+43. 
```


In [20]:
class DetectionValue(): 
    
    FLOAT_LITERAL = r"[0-9]+(\.[0-9]*)?([Ee][+-]?[0-9]+)?"; 
    LESS_SIGN = r"\<?"; 
    LOWER_LIMIT_FILTER = re.compile(
        r"\A(?P<lt>{lt})(?P<lim>{fp})\Z".format(
            lt=LESS_SIGN, fp=FLOAT_LITERAL
        )
    ); 
    
    @classmethod
    def parse(cls, data: str) -> float: 
        match = cls.LOWER_LIMIT_FILTER.match(data); 
        if not match: 
            raise ValueError(
                "Invalid data detected while parsing: {data}".format(data=data)
            ); 
        match_grp = match.groupdict(); 
        num = float(match_grp["lim"]); 
        if match_grp["lt"]: 
            num /= 2; 
        return num; 
    

In [21]:
epoch = datetime.datetime(1970, 1, 1); 
def rec_item_norm(item: list, env=None) -> list: 
    item_norm = item.copy(); 
    if env == "river": 
        idx_fld_date = 3; 
    elif env == "marine": 
        idx_fld_date = 2; 
    dates = datetime.datetime.fromisoformat(item[idx_fld_date]); 
    date_diff = dates - epoch; 
    item_norm[idx_fld_date] = date_diff.days; 
    for idx, val in enumerate(item): 
        if env == "river": 
            if idx == 4: 
                item_norm[idx] = int(val); 
                continue; 
            elif idx <= 3: 
                continue; 
        if env == "marine": 
            if idx == 3: 
                item_norm[idx] = int(val); 
                continue; 
            elif idx <= 4: 
                continue; 
        if val == str() or val == "null" or val.casefold() == "n/a": 
            item_norm[idx] = None; 
            continue; 
        else: 
            item_norm[idx] = DetectionValue.parse(val); 
    item_norm.insert(idx_fld_date + 1, dates.year); 
    item_norm.insert(idx_fld_date + 2, dates.month); 
    item_norm.insert(idx_fld_date + 3, dates.day); 
    return item_norm; 

In [22]:
wq_riv_items_norm = list(
    rec_item_norm(rec, env="river") for rec in wq_riv_items
); 
wq_mar_items_norm = list(
    rec_item_norm(rec, env="marine") for rec in wq_mar_items
); 

## `csv`提取结果合并入库

### 字段声明与二维表建立
在数据库中建立两个二维表, 分别存储河流和海洋水质监测数据, 其字段名称采用前述[字段重命名](#字段重命名)的结果. 

#### 字段名称与类型声明

In [23]:
wq_riv_field_descr_iter = zip(
    wq_riv_fields_norm, 
    it.chain(
        ("TEXT", ) * 3, 
        ("REAL", ), 
        ("INTEGER", ) * 4, 
        it.repeat("REAL")
    )
); 
wq_riv_field_descr_sql = ",\x20".join(
    "{fld}\x20{cls}".format(
        fld=fld, cls=cls
    ) for (fld, cls) in wq_riv_field_descr_iter
); 
wq_riv_field_param_sql = "({0})".format(
    ",\x20".join("?" for _ in wq_riv_fields_norm)
); 

wq_mar_field_descr_iter = zip(
    wq_mar_fields_norm, 
    it.chain(
        ("TEXT", ) * 2, 
        ("REAL", ), 
        ("INTEGER", ) * 4, 
        ("TEXT", ), 
        it.repeat("REAL")
    )
); 
wq_mar_field_descr_sql = ",\x20".join(
    "{fld}\x20{cls}".format(
        fld=fld, cls=cls
    ) for (fld, cls) in wq_mar_field_descr_iter
); 
wq_mar_field_param_sql = "({0})".format(
    ",\x20".join("?" for _ in wq_mar_fields_norm)
); 

print(
    wq_riv_field_descr_sql, wq_riv_field_param_sql, 
    wq_mar_field_descr_sql, wq_mar_field_param_sql, 
    sep="\n"
)

water_control_zone TEXT, river TEXT, station TEXT, dates REAL, year_obsv INTEGER, month_obsv INTEGER, day_obsv INTEGER, sample_no INTEGER, _day_biochemical_oxygen_demand REAL, aluminium REAL, ammonia_nitrogen REAL, anionic_surfactants REAL, antimony REAL, arsenic REAL, barium REAL, beryllium REAL, boron REAL, cadmium REAL, chemical_oxygen_demand REAL, chloride REAL, chlorophyll_a REAL, chromium REAL, conductivity REAL, copper REAL, cyanide REAL, dissolved_oxygen REAL, dissolved_oxygen_2 REAL, faecal_coliforms REAL, flow REAL, fluoride REAL, free_hydrogen_sulphide REAL, iron REAL, lead REAL, manganese REAL, mercury REAL, molybdenum REAL, nickel REAL, nitrate_nitrogen REAL, nitrite_nitrogen REAL, oil_and_grease REAL, orthophosphate_phosphorus REAL, ph REAL, phaeopigment REAL, salinity REAL, silica REAL, silver REAL, sulphide REAL, suspended_solids REAL, thallium REAL, total_kjeldahl_nitrogen REAL, total_organic_carbon REAL, total_phosphorus REAL, total_solids REAL, total_volatile_solids 

#### 建立数据库连接并创建表格

In [24]:
wq_items_sqlite = sqlite3.connect(os.sep.join(
    [nb_dir, "wq_spot_value.sqlite"]
) ); 

In [25]:
wq_items_sqlite.execute(
    """
    DROP TABLE If EXISTS info_spot_value_river; 
    """
); 
wq_items_sqlite.execute(
    """
    CREATE TABLE info_spot_value_river (
        {field_descr}
    ); 
    """.format(field_descr=wq_riv_field_descr_sql)
); 
wq_items_sqlite.commit(); 

In [26]:
wq_items_sqlite.execute(
    """
    DROP TABLE If EXISTS info_spot_value_marine; 
    """
); 
wq_items_sqlite.execute(
    """
    CREATE TABLE info_spot_value_marine (
        {field_descr}
    ); 
    """.format(field_descr=wq_mar_field_descr_sql)
); 
wq_items_sqlite.commit(); 

### 数据的插入

In [27]:
wq_items_sqlite.executemany(
    """
    INSERT Into info_spot_value_river
        VALUES {param}
    """.format(param=wq_riv_field_param_sql), 
    wq_riv_items_norm
); 
wq_items_sqlite.executemany(
    """
    INSERT Into info_spot_value_marine
        VALUES {param}
    """.format(param=wq_mar_field_param_sql), 
    wq_mar_items_norm
); 
wq_items_sqlite.commit(); 

### 数据库保存与关闭

In [28]:
wq_items_sqlite.execute("VACUUM"); 
wq_items_sqlite.commit(); 
wq_items_sqlite.close(); 